# Understanding the Roman WFI ePSF Reference File

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.1".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.
    

## Introduction
The purpose of this notebook is to understand the content and purpose of the **Empirical PSF** (`PSF` / `ePSF`) reference file.

The ePSF reference file contains high-fidelity empirical point spread function stamps. It is used by the `source_catalog` step for accurate photometry and morphology measurements.

More details about this and other reference files can be found in the [Reference File Information](https://roman-pipeline.readthedocs.io/en/stable/roman/references_general/index.html).

### Local Run Settings

If you want to run the notebook in your local machine, refer to the information in [local installation](../../markdown/local-run.md) instructions before proceeding with the notebook. The instructions provide important information about setting up your environment and installing dependencies.

## Imports
Libraries used:
- *astropy* for image normalization
- *copy* for making copies of Python objects
- *crds* for access to calibration reference files
- *matplotlib* and *mpl_toolkits* for plotting images
- *numpy* for array manipulation
- *roman_datamodels* for opening Roman WFI ASDF files
- *os* for operating system functions

In [ ]:
import os
from astropy.visualization import simple_norm
import copy

import matplotlib.pyplot as plt
from matplotlib import colors, colormaps as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import roman_datamodels as rdm

### The Calibration Reference Data System (CRDS)

The reference files, developed and validated by STScI’s Science Operations Center, are continually updated as new WFI data become available. For more information about how CRDS works and how it assigns the most appropriate reference file for each calibration step, refer to the notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb). 

**IMPORTANT NOTE:** Reference files are a work in progress and will be updated several times before Roman launch. If you notice irregularities or missing information, please understand that they may be a known issue. If you have questions, please contact the [Roman Help Desk](https://romanhelp.stsci.edu).

In [ ]:
import crds

Now let's dive into this reference file type.

### Empirical PSF Reference File

The ePSF reference file contains empirical PSF stamps (including versions with and without inter-pixel capacitance). It is used for source detection and photometry.

For more details, see the [romancal documentation](https://roman-pipeline.readthedocs.io/en/stable/roman/references_general/psf_reffile.html#psf-reference-file).


Before proceeding, let's check the environmental variables set for CRDS

In [ ]:
print(f"CRDS server location: {os.environ.get('CRDS_SERVER_URL')}")
print(f"CRDS context file: {os.environ.get('CRDS_CONTEXT')}")

If we want to change the context, we can do it in the next cell. In this case, we choose context `roman_0055.pmap`.

In [ ]:
os.environ['CRDS_CONTEXT']='roman_0055.pmap'

### Retrieving Reference Files

As you run the exposure pipeline, the most up-to-date reference files will be automatically selected for each step. However, if you would like to use a specific reference file, retrieve it using the `CRDS` Python API and feed it to the Exposure Level and Mosaic Pipeline, see the notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb) for more details. 

In CRDS the PSF reference files are refered as the Empirial PSF or ePSF. The keywords that will identify the best reference file to use are:

- ROMAN.META.INSTRUMENT.NAME
- ROMAN.META.INSTRUMENT.OPTICAL_ELEMENT
- ROMAN.META.EXPOSURE.START_TIME

These keywords may be combined into a single dictionary to find and download the file using `crds.getreferences()`.  

In [ ]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.OPTICAL_ELEMENT': 'F158',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getreferences(meta, reftypes=['epsf'], observatory='roman')
ref_files

### Examining Reference Files

Reference files use `roman_datamodels` just like WFI science data products and can be accessed in the same way (see the tutorial [Working with ASDF](../working_with_asdf/working_with_asdf.ipynb) for more information). Let's take a closer look at the files we retrieved from our `crds.getreferences()` example:

In [ ]:
epsf = rdm.open(ref_files['epsf'])
epsf.info()

We see that the gain reference file contains metadata plus the `data`,  a 2D array with the gain value (electrons per DN) for each pixel.

### Basic Statistics

Now lets get some basic statistics on the cube (or a representative slice)

In [ ]:
for name in ['psf', 'extended_psf', 'psf_noipc', 'extended_psf_noipc']:
    if hasattr(epsf, name):
        data = getattr(epsf, name)
        print(f"\n{name} shape: {data.shape}")
        print(f"  Min: {data.min():.2e}   Max: {data.max():.2e}")
        print(f"  Mean: {data.mean():.2e}")

In [ ]:
print("PSF array shape:", epsf.psf.shape)
print("Suggested slices to try:")

for d0 in range(epsf.psf.shape[0]):
    for d1 in range(epsf.psf.shape[1]):
        for d2 in [0, 4, 8]:   # edges and middle
            print(f"  epsf.psf[{d0}, {d1}, {d2}]")

### Visualization

Let's check this reference file

In [ ]:
fig = plt.figure(figsize=(14, 10))

# Extended PSF (large stamp)
if hasattr(epsf, 'extended_psf'):
    ax1 = plt.subplot(2, 2, 1)
    data = epsf.extended_psf
    norm = simple_norm(data, stretch='log', percent=99.5)
    im1 = ax1.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax1.set_title('Extended PSF')
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

# Extended PSF no IPC
if hasattr(epsf, 'extended_psf_noipc'):
    ax2 = plt.subplot(2, 2, 2)
    data = epsf.extended_psf_noipc
    norm = simple_norm(data, stretch='log', percent=99.5)
    im2 = ax2.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax2.set_title('Extended PSF (no IPC)')
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

# Central slice of the small PSF stamp (e.g., central oversampled PSF)
if hasattr(epsf, 'psf'):
    ax3 = plt.subplot(2, 2, 3)
    data = epsf.psf[1, 1, 4]          # middle slice (adjust indices as needed)\
    data = epsf.psf[2,2,0]
    norm = simple_norm(data, stretch='log', percent=99.5)
    im3 = ax3.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax3.set_title('PSF Stamp (central slice)')
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

# Same for no IPC
if hasattr(epsf, 'psf_noipc'):
    ax4 = plt.subplot(2, 2, 4)
    data = epsf.psf_noipc[1, 1, 4]
    norm = simple_norm(data, stretch='log', percent=99.5)
    im4 = ax4.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax4.set_title('PSF Stamp no IPC (central slice)')
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(14, 10))

# Extended PSF (large stamp)
if hasattr(epsf, 'extended_psf'):
    ax1 = plt.subplot(3, 2, 1)
    data = epsf.extended_psf
    norm = simple_norm(data, stretch='log', percent=99.5)
    im1 = ax1.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax1.set_title('Extended PSF')
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

# Extended PSF no IPC
if hasattr(epsf, 'extended_psf_noipc'):
    ax2 = plt.subplot(3,2,2)
    data = epsf.extended_psf_noipc
    norm = simple_norm(data, stretch='log', percent=99.5)
    im2 = ax2.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax2.set_title('Extended PSF (no IPC)')
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

# Central slice of the small PSF stamp
if hasattr(epsf, 'psf'):
    ax3 = plt.subplot(3,2, 3)
    data = epsf.psf[1, 1, 4]          # middle slice (adjust indices as needed)
    norm = simple_norm(data, stretch='log', percent=99.5)
    im3 = ax3.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax3.set_title('PSF Stamp (central slice)')
    plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

# Central slice of the small PSF stamp
if hasattr(epsf, 'psf'):
    ax5 = plt.subplot(3,2,5)
    data = epsf.psf[0,0, 0]          # middle slice (adjust indices as needed)
    norm = simple_norm(data, stretch='log', percent=99.5)
    im5 = ax5.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax5.set_title('PSF Stamp (No defocus [0,0,0])')
    plt.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04)

# Central slice of the small PSF stamp
if hasattr(epsf, 'psf'):
    ax6 = plt.subplot(3,2,6)
    data = epsf.psf[2, 2, 8]          # middle slice (adjust indices as needed)
    norm = simple_norm(data, stretch='log', percent=99.5)
    im6 = ax6.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax6.set_title('PSF Stamp (Defocused [2,2,8])')
    plt.colorbar(im6, ax=ax6, fraction=0.046, pad=0.04)

# Same for no IPC
if hasattr(epsf, 'psf_noipc'):
    ax4 = plt.subplot(3, 2, 4)
    data = epsf.psf_noipc[1, 1, 4]
    norm = simple_norm(data, stretch='log', percent=99.5)
    im4 = ax4.imshow(data, cmap='viridis', norm=norm, origin='lower')
    ax4.set_title('PSF Stamp no IPC (central slice)')
    plt.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

**Note:** The small PSF stamps have shape `(3, 3, 9, 361, 361)`. The first three dimensions are parameters (defocus, other conditions, oversampling). You can explore other slices by changing the indices `[1, 1, 4]`.


## About this Notebook
**Author:** R. Diaz

**Updated On:** 2026-07-06

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>